# imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# config

In [2]:
ASSET = "BTC"
INTERVAL = "1h"

# mlflow

In [3]:
mlflow.set_tracking_uri("http://localhost:5000")

experiment_name = f"{ASSET}_{INTERVAL}_Direction_Prediction"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='/mlruns/9', creation_time=1778237968335, experiment_id='9', last_update_time=1778237968335, lifecycle_stage='active', name='BTC_1h_Direction_Prediction', tags={}, trace_location=None, workspace='default'>

# load data 

In [4]:
train_df = pd.read_parquet('../../../data/processed/train_btc_1h.parquet')
test_df = pd.read_parquet('../../../data/processed/test_btc_1h.parquet')

In [5]:
print(f"Training shapes: {train_df.shape}")
print(f"Testing shapes: {test_df.shape}")

Training shapes: (42753, 41)
Testing shapes: (10689, 41)


# separate features from target

In [6]:
features = [col for col in train_df.columns if col not in ['target_1h', 'target_direction']]

X_train = train_df[features]
y_train = train_df['target_direction']

X_test = test_df[features]
y_test = test_df['target_direction']

print("done splitttign into X and y!")

done splitttign into X and y!


# mlflow experiment tracking

# logistic regression

In [20]:
with mlflow.start_run(run_name=f"Baseline_LogisticRegression_{ASSET}_{INTERVAL}"):
    
    print(f"Training the baseline model for {ASSET} ({INTERVAL})...")
    
    model_lr = LogisticRegression(random_state=42, max_iter=1000)
    model_lr.fit(X_train, y_train)
    
    predictions_lr = model_lr.predict(X_test)
    
    acc = accuracy_score(y_test, predictions_lr)
    print(f"\nBaseline Accuracy: {acc * 100:.2f}%")
    
    print("\nReport")
    print(classification_report(y_test, predictions_lr))

    mlflow.log_param("asset", ASSET)
    mlflow.log_param("interval", INTERVAL)
    
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("features_count", len(features))
    
    mlflow.log_metric("accuracy", acc)
    
    mlflow.sklearn.log_model(model_lr, "logistic_regression_model")
    
    print(f"\n baseline model linear regression experiment loggged Run finished for {ASSET} {INTERVAL}.")

Training the baseline model for BTC (1h)...

Baseline Accuracy: 52.26%

Report
              precision    recall  f1-score   support

           0       0.52      0.54      0.53      5327
           1       0.52      0.51      0.52      5362

    accuracy                           0.52     10689
   macro avg       0.52      0.52      0.52     10689
weighted avg       0.52      0.52      0.52     10689



2026/05/08 12:22:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 12:22:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



 experiment loggged Run finished for BTC 1h.
🏃 View run Baseline_LogisticRegression_BTC_1h at: http://localhost:5000/#/experiments/9/runs/8a9e3b139ad34e3cb7ee1ec3f29a3be8
🧪 View experiment at: http://localhost:5000/#/experiments/9


# random forest

In [22]:
with mlflow.start_run(run_name=f"RandomForest_{ASSET}_{INTERVAL}"):
    
    print(f"Training Random Forest for {ASSET} ({INTERVAL})...")
    
    model_rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model_rf.fit(X_train, y_train)
    
    predictions_rf = model_rf.predict(X_test)
    
    acc = accuracy_score(y_test, predictions_rf)
    print(f"\nRandom Forest Accuracy: {acc * 100:.2f}%")
    
    print("\nReport")
    print(classification_report(y_test, predictions_rf))

    mlflow.log_param("asset", ASSET)
    mlflow.log_param("interval", INTERVAL)
    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("features_count", len(features))
    
    mlflow.log_metric("accuracy", acc)
    mlflow.sklearn.log_model(model_rf, "random_forest_model")
    
    print(f"\randomforest logged and run finished for {ASSET} {INTERVAL}.")

Training Random Forest for BTC (1h)...

Random Forest Accuracy: 51.97%

Report
              precision    recall  f1-score   support

           0       0.51      0.64      0.57      5327
           1       0.53      0.40      0.46      5362

    accuracy                           0.52     10689
   macro avg       0.52      0.52      0.51     10689
weighted avg       0.52      0.52      0.51     10689



2026/05/08 12:27:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 12:27:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


andomforest logged and run finished for BTC 1h.
🏃 View run RandomForest_BTC_1h at: http://localhost:5000/#/experiments/9/runs/099e7598e8f646aa8c57b4a0ae795b8f
🧪 View experiment at: http://localhost:5000/#/experiments/9


# XGBoost

In [8]:
with mlflow.start_run(run_name=f"XGBoost_{ASSET}_{INTERVAL}"):
    
    print(f"Training XGBoost for {ASSET} ({INTERVAL})...")
    
    model_xgb = xgb.XGBClassifier(
        n_estimators=100, 
        max_depth=4, 
        learning_rate=0.05, 
        random_state=42,
        eval_metric='logloss'
    )
    
    model_xgb.fit(X_train, y_train)
    
    predictions_xgb = model_xgb.predict(X_test)
    
    acc = accuracy_score(y_test, predictions_xgb)
    print(f"Accuracy: {acc * 100:.2f}%")
    
    print("\nReport")
    print(classification_report(y_test, predictions_xgb))

    mlflow.log_param("asset", ASSET)
    mlflow.log_param("interval", INTERVAL)
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 4)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("features_count", len(features))
    
    mlflow.log_metric("accuracy", acc)
    
    mlflow.xgboost.log_model(model_xgb, "xgboost_model")
    
    print(f"\nXGBoost logged and run finished for {ASSET} {INTERVAL} .")

Training XGBoost for BTC (1h)...
Accuracy: 53.12%

Report
              precision    recall  f1-score   support

           0       0.53      0.59      0.56      5327
           1       0.54      0.47      0.50      5362

    accuracy                           0.53     10689
   macro avg       0.53      0.53      0.53     10689
weighted avg       0.53      0.53      0.53     10689



2026/05/08 12:42:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



XGBoost logged and run finished for BTC 1h .
🏃 View run XGBoost_BTC_1h at: http://localhost:5000/#/experiments/9/runs/3f4396dd296d495aae0297ec3a2f4925
🧪 View experiment at: http://localhost:5000/#/experiments/9
